# OpenClaw

## Setup

The below steps apply to the Linux operating system. Refer to the [documentation](https://docs.openclaw.ai/start/getting-started) for other operating systems.

#### Install OpenClaw

In [ ]:
!curl -fsSL https://openclaw.ai/install.sh | bash

#### Install the openclaw service.  
Note that this skips the onboarding, we will configure the settings manually.

In [ ]:
!openclaw gateway --install-daemon

#### Verify gateway is running

In [ ]:
!openclaw gateway status

## Configure OpenClaw

#### Backup the default openclaw config.

In [ ]:
!cp $HOME/.openclaw/openclaw.json $HOME/.openclaw/openclaw.json.default

#### Specify model attributes

Replace `<your_api_key>` with your api key in the json below.  
Replace `models` in `$HOME/.openclaw/openclaw.json` with the completed json

```json
"models": {
    "mode": "merge",
    "providers": {
    "nvidia-nim": {
        "baseUrl": "https://integrate.api.nvidia.com/v1",
        "apiKey": "<your_api_key>",
        "api": "openai-completions",
        "models": [
            {
                "id": "nvidia/nemotron-3-super-120b-a12b",
                "name": "nvidia/nemotron-3-super-120b-a12b (Custom Provider)",
                "reasoning": false,
                "input": [
                    "text"
                ],
                "cost": {
                "input": 0,
                "output": 0,
                "cacheRead": 0,
                "cacheWrite": 0
                },
                "contextWindow": 100000,
                "maxTokens": 4096
            }
        ]
    }
    }
}
```

#### Specify agents attributes

Replace `<your_home_directory>` with your actual home directory in the json below.   
Replace `agents` in `$HOME/.openclaw/openclaw.json` with the completed json.

```json
"agents": {
    "defaults": {
      "model": {
        "primary": "nvidia-nim/nvidia/nemotron-3-super-120b-a12b"
      },
      "models": {
        "nvidia-nim/nvidia/nemotron-3-super-120b-a12b": {}
      },
      "workspace": "<your_home_directory>/.openclaw/workspace"
    },
    "list": [
      {
        "id": "main"
      },
      {
        "id": "music-store-agent",
        "name": "music-store-agent",
        "workspace": "<your_home_directory>/.openclaw/workspace-music-store-agent",
        "agentDir": "<your_home_directory>/.openclaw/agents/music-store-agent/agent",
        "model": "nvidia/nemotron-3-super-120b-a12b",
        "default": true
      }
    ]
}
```

#### Define the music store agent

Replace contents of `soul.md` file with the following.

```
# Music Store Agent

You are a music store agent. Do not invent anything. All data comes from skills.

## Allowed Skills
- music-store-assistant
```

#### Create agent skill

Create `SKILL.md` file and input the below contents.

```
---
name: music-store-assistant
description: lookup information on tracks,artists,abums and process refunds.
---

### Query the music store assistant

Replace <query> with question from user/agent. Do not remove any information.

```bash
curl -X POST http://localhost:8001/generate \
  -H "Content-Type: application/json" \
  -d $'{
    "input_message": <query>
  }'
```
```

#### Allow `curl` command to be run by the music store agent.  
Replace `agents` in `$HOME/.openclaw/exec-approvals.json` with the below json.

```json
"agents": {
    "music-store-agent": {
      "security": "allowlist",
      "ask": "on-miss",
      "allowlist": [
        {
          "pattern": "/usr/bin/curl"
        }
      ]
    }
  }
```

#### Restart openclaw gateway to effect the changes made above

In [ ]:
!openclaw gateway restart

#### Serve the completed workflow from [assignment 4 - nemo agent toolkit](../../challenge/nemo_agent_toolkit/) as a web server

In [ ]:
!nat serve --port 8001 --config_file ../../challenge/nemo_agent_toolkit/workflaw.yaml

#### Test the web endpoint

In [ ]:
!curl -X POST http://localhost:8001/generate \
  -H "Content-Type: application/json" \
  -d $'{
    "input_message": "my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice IDs related to the artist Led Zeppelin"
  }'

#### Run OpenClaw TUI

In [ ]:
!openclaw tui

#### Ask the following question to the music-store-agent

`my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice IDs related to the artist Led Zeppelin`

Expected output:
##TODO

#### View session traces

In [ ]:
!cat $HOME/.openclaw/agents/music-store-agent/sessions/<sessionId>.jsonl